# 第 7 课：Model-based RL、误差传播与代理模型奖励可信度

**学习目标**：让“代理模型取代高成本 CFD 环境”从一句想法变成可以量化检验的方案。

真实动力学 $s_{t+1}=F(s_t,a_t)$，学习模型 $\hat F_\psi$。训练数据来自真实环境，RL 则可能反复在 $\hat F_\psi$ 中 rollout。

单步测试误差小不代表闭环安全：

$$e_{t+1}=\hat F_\psi(\hat s_t,a_t)-F(s_t,a_t).$$

右侧同时包含**模型自身误差**与之前状态预测误差的传播。控制器优化的是整个闭环回报，可能专门寻找误差最大的动作区域。

## 7.1 构造一个略有非线性的真实系统，拟合线性代理

真实系统是教学代替品，与 Navier–Stokes 无关。训练集的动作在 [-1,1] 以内；闭环策略却可能向模型外推区移动。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(88)
def true_dynamics(x, a):
    return .85*x + .27*a + .09*np.tanh(3*x) + .025*(a**3)

N = 1200
x_train = rng.uniform(-1.,1.,N)
a_train = rng.uniform(-1.,1.,N)
y_train = true_dynamics(x_train,a_train) + rng.normal(0,.01,N)

def features(x,a):
    return np.stack([np.ones_like(x),x,a],axis=-1)

coef = np.linalg.lstsq(features(x_train,a_train), y_train, rcond=None)[0]
def model_dynamics(x,a):
    return coef[0]+coef[1]*x+coef[2]*a

print('Fitted linear surrogate coefficients:', np.round(coef,4))
print('Training one-step MSE:', np.mean((model_dynamics(x_train,a_train)-y_train)**2))

## 7.2 区分 IID 测试误差、OOD 误差与多步轨迹误差

In [ ]:
def mse(x,a):
    return float(np.mean((model_dynamics(x,a)-true_dynamics(x,a))**2))

iid_x = rng.uniform(-1.,1.,1000); iid_a=rng.uniform(-1.,1.,1000)
ood_x = rng.uniform(-2.,2.,1000); ood_a=rng.uniform(-2.5,2.5,1000)
print('IID one-step MSE:', round(mse(iid_x,iid_a),6))
print('OOD one-step MSE:', round(mse(ood_x,ood_a),6))

T=45
a_seq = 1.6*np.sin(np.linspace(0,8,T))
x_true=[0.7]; x_model=[0.7]
for a in a_seq:
    x_true.append(float(true_dynamics(x_true[-1],a)))
    x_model.append(float(model_dynamics(x_model[-1],a)))
plt.figure(figsize=(7,3))
plt.plot(x_true,label='true trajectory')
plt.plot(x_model,label='surrogate rollout',linestyle='--')
plt.legend(); plt.xlabel('Time step'); plt.ylabel('State'); plt.show()

## 7.3 验证奖励排名，而不只看流场 MSE

假设目标是把 $x$ 压低，同时限制动作能耗：

$$r_t=-x_{t+1}^2-0.04a_t^2.$$

我们比较不同反馈增益 $a_t=\mathrm{clip}(-k x_t,-2.5,2.5)$ 在真实动力学与代理动力学下的回报排名。实际研究中可以把 $x$ 替换为升力/阻力相关的状态变量。

In [ ]:
def rollout_score(dynamics, k, starts=np.linspace(-1.5,1.5,41), T=45):
    scores=[]
    for x0 in starts:
        x=float(x0); total=0.
        for t in range(T):
            a=float(np.clip(-k*x,-2.5,2.5))
            x=float(dynamics(x,a))
            total += -x*x - .04*a*a
        scores.append(total)
    return float(np.mean(scores))

gains=np.linspace(-1.,4.,40)
real_scores=np.array([rollout_score(true_dynamics,k) for k in gains])
model_scores=np.array([rollout_score(model_dynamics,k) for k in gains])
plt.figure(figsize=(7,3))
plt.plot(gains,real_scores,label='Real system')
plt.plot(gains,model_scores,label='Learned surrogate',linestyle='--')
plt.xlabel('Feedback gain k'); plt.ylabel('Mean closed-loop return')
plt.legend(); plt.show()
print('surrogate-best gain:',round(gains[np.argmax(model_scores)],2))
print('real-best gain:     ',round(gains[np.argmax(real_scores)],2))
print('rank correlation:',round(float(np.corrcoef(np.argsort(np.argsort(real_scores)),np.argsort(np.argsort(model_scores)))[0,1]),3))

## 7.4 模型不确定性：Bootstrap ensemble 的直觉

独立重采样训练数据并重新拟合模型，计算模型之间的预测方差。它只是一种粗略的**认知不确定性信号**，并不能保证检测全部外推误差，尤其当模型家族有同样的结构性偏差时。

In [ ]:
ensemble=[]
for i in range(12):
    ids=rng.integers(0,N,N)
    ensemble.append(np.linalg.lstsq(features(x_train[ids],a_train[ids]),y_train[ids],rcond=None)[0])
ensemble=np.asarray(ensemble)
test_action=np.linspace(-3,3,160)
test_state=np.full_like(test_action,0.5)
preds = np.array([features(test_state,test_action)@c for c in ensemble])
mean_pred=preds.mean(axis=0); std_pred=preds.std(axis=0)
plt.figure(figsize=(7,3))
plt.plot(test_action,true_dynamics(test_state,test_action),label='True')
plt.plot(test_action,mean_pred,label='Ensemble mean')
plt.fill_between(test_action,mean_pred-2*std_pred,mean_pred+2*std_pred,alpha=.25,label='±2 std (not calibrated)')
plt.axvspan(-1,1,alpha=.08,label='Training action support')
plt.xlabel('Action'); plt.legend(); plt.show()

## 7.5 反例：代理奖励在外推区被控制器利用

以下是**人为构造的失败案例**，不是实际 CFD 或训练结果。真实奖励和代理预测奖励在训练动作区间 $a\in[-1,1]$ 内完全一致；代理在区间外高估高幅值动作的收益。最大化代理奖励的控制器可能追逐一个物理上不存在的收益。

In [ ]:
actions_grid=np.linspace(-3.,3.,301)
def true_control_reward(a):
    return .12*a - .09*a*a

def optimistic_surrogate_reward(a):
    # 仅在训练动作支持范围之外引入人为的正向偏差
    return true_control_reward(a) + .15*np.maximum(np.abs(a)-1.,0.)**2

true_r=true_control_reward(actions_grid)
hat_r=optimistic_surrogate_reward(actions_grid)
i_true=int(np.argmax(true_r)); i_model=int(np.argmax(hat_r))
print('True-best a:', round(actions_grid[i_true],2), 'true reward:',round(true_r[i_true],3))
print('Model-best a:',round(actions_grid[i_model],2),
      'predicted reward:',round(hat_r[i_model],3),
      'true reward:',round(true_r[i_model],3))
plt.figure(figsize=(7,3))
plt.plot(actions_grid,true_r,label='True reward')
plt.plot(actions_grid,hat_r,label='Biased surrogate reward',linestyle='--')
plt.axvspan(-1,1,alpha=.08,label='Training action support')
plt.legend(); plt.xlabel('Action'); plt.ylabel('Reward'); plt.show()
assert true_r[i_model] < true_r[i_true]

## 7.6 可微代理模型与 PPO：两条不同优化路线

JAX 还能对可微的代理动力学展开多个时间步，直接对控制策略参数求梯度。这更接近 **differentiable control / model predictive optimization**；PPO 则通过环境采样和 log-probability 学习，环境本身无需可微。

即使代理可微，沿代理反传的梯度仍可能因为模型偏差、长期动力学敏感性而不可信，所以不能以“可微”替代独立 CFD 物理验证。

In [ ]:
import jax
import jax.numpy as jnp
c=jnp.asarray(coef)

def differentiable_rollout_loss(k):
    def step(x, _):
        action=jnp.clip(-k*x,-2.5,2.5)
        next_x=c[0]+c[1]*x+c[2]*action
        reward=-(next_x**2+.04*action**2)
        return next_x,reward
    _, rewards=jax.lax.scan(step,jnp.array(1.,dtype=jnp.float32),None,length=35)
    return -jnp.sum(rewards)

k0=jnp.array(1.0)
print('Surrogate predicted loss:',float(differentiable_rollout_loss(k0)))
print('Gradient dLoss/dGain:',float(jax.grad(differentiable_rollout_loss)(k0)))

## 7.7 给毕设的实验清单

1. 比较单步误差、自由预测多步误差、给定真实控制序列的 rollout 误差。
2. 更重要：比较**相同控制策略下，代理环境与真实 CFD 的平均回报、阻力变化及策略排名**。
3. 显式记录代理训练数据的 Reynolds 数、动作幅值、频率、传感器位置与初始条件覆盖。
4. 优化过程中定期拿候选策略回到真实 CFD，补充高误差区域数据（主动学习 / Dyna 思路）。
5. 分开报告模型不确定性、控制不确定性、统计置信区间。

### 练习

把线性 surrogate 换成两层 JAX MLP；对比短 rollout 与长 rollout 的预测误差以及最终策略排名一致性。